In [ ]:
import os
from dotenv import load_dotenv
from crewai import Agent, Task, Crew, Process, LLM
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from crewai.tools import tool
from langchain_community.document_loaders import DirectoryLoader, PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from config.config import API_KEY
load_dotenv()

False

In [ ]:
# ====================== CONFIG: EVERYTHING ON YOUR OTHER DRIVE ======================
BASE_DIR = "D:/AI_Data/Computer/Agent-WQB/"
WQB_CONSULTANT_FORUM_PDF_PATH = "D:/AI_Data/WQB-Consultant-Data/Forums/wqb_consultant_pdf/"
CHROMA_DIR = BASE_DIR + "quant_forum_chroma/"
BGEM3_DIR = BASE_DIR + "quant_forum_bgem3/"
HF_CACHE_DIR = BASE_DIR + "cache/hf/"

# Redirect caches so nothing touches C:
# os.environ["HF_HOME"] = HF_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = HF_CACHE_DIR

# Create the cache folder if it doesn't exist yet
os.makedirs(CHROMA_DIR, exist_ok=True)
os.makedirs(HF_CACHE_DIR, exist_ok=True)
os.makedirs(BGEM3_DIR, exist_ok=True)

# ====================== YOUR GEMINI CLIENT ======================
"""
CrewAI’s Agent class no longer accepts a raw langchain_openai.ChatOpenAI object directly in the llm= parameter (Pydantic validation fails).
Instead, you must use CrewAI’s own LLM wrapper (it internally uses LiteLLM and works perfectly with your reverse-engineered Gemini proxy).
"""
# llm = ChatOpenAI(
#     model="gemini-3.1-pro",
#     base_url="http://100.127.232.48:8000/v1",
#     api_key=API_KEY
    # temperature=0.7,
    # max_tokens=8192
# )

# ==================================== CELL: CrewAI LLM (Fixed for your proxy) ====================================

# Use your exact proxy settings
llm = LLM(
    model="gemini-3.1-pro",   # ← change if your proxy uses a different model name
    base_url="http://100.127.232.48:8000/v1",
    api_key=API_KEY,
    temperature=0.6,          # slightly lower = more stable
    max_tokens=8192,
    timeout=180,              # give it more time
    max_retries=5,            # extra retries
)

In [4]:
# llm.invoke([{"role": "user", "content": "Hello, how are you?"}]) # Test the client (for openai-compatible LLMs)
# llm.call("Hello, how are you?") # Simpler test method (for crewai's LLM wrapper)

In [ ]:
# ====================== EMBEDDINGS ======================

# Use if you have access to Gemini Embeddings, which are optimized for use with Gemini models. Otherwise, HuggingFaceEmbeddings is a solid choice.
# embeddings = OpenAIEmbeddings(
#     model="gemini-embedding-001",
#     base_url="http://100.127.232.48:8000/v1",
#     api_key=API_KEY
# )

# embeddings = HuggingFaceEmbeddings(
#     model_name="nomic-ai/nomic-embed-text-v1.5",
#     cache_folder=HF_CACHE_DIR,  # Use the custom cache directory
#     model_kwargs={"device": "cpu"},           # force CPU (your low GPU setup)
#     encode_kwargs={"normalize_embeddings": True},  # best for Chroma similarity search
#     show_progress=True
# )

In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3", # excellent for Chinese
    cache_folder=HF_CACHE_DIR,  # Use the custom cache directory
    model_kwargs={"device": "cpu"},           # force CPU (your low GPU setup)
    encode_kwargs={"normalize_embeddings": True},  # best for Chroma similarity search
    show_progress=True
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [7]:
# ==================================== Ingest Forum (PDFs) (ONE-TIME) ====================================
def ingest_forum():
    # os.makedirs(FORUM_DATA_DIR, exist_ok=True)
    
    loader = DirectoryLoader(
        WQB_CONSULTANT_FORUM_PDF_PATH,
        glob="**/*.pdf",                    # ← changed for your PDFs
        loader_cls=PyPDFLoader,
        silent_errors=False,
        show_progress=True
    )
    docs = loader.load()
    
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=400)
    splits = text_splitter.split_documents(docs)

    vectorstore = Chroma.from_documents(
        documents=splits,
        embedding=embeddings,               # defined in above cell
        persist_directory=BGEM3_DIR
    )
    print(f"✅ Ingested {len(splits)} chunks from {len(docs)} PDF files")
    return vectorstore

# First run only:
# vectorstore = ingest_forum()

In [8]:
vectorstore = Chroma(persist_directory=BGEM3_DIR, embedding_function=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 8})

In [9]:
# ====================== FORUM SEARCH TOOL ======================
@tool("Forum_Search")
def forum_search(query: str) -> str:
    """Search the entire quant forum archive."""
    docs = retriever.invoke(query)
    return "\n\n---\n\n".join([doc.page_content for doc in docs])

# ====================== AGENTS (Your Quant Research Team) ======================
researcher = Agent(
    role="WorldQuant Forum Researcher & Master Analyst",
    goal="You MUST use the forum_search tool to extract real content from the WorldQuant Brain Consultant Forum PDFs. Never answer from general knowledge.",
    backstory="""You are a veteran WorldQuant Brain consultant who has read every single PDF in the forum archive.
    Your only source of truth is the `forum_search` tool. You always call it first.
    You never say you don't have access — you always use the tool.""",
    tools=[forum_search],
    llm=llm,
    verbose=True,
    allow_delegation=False
)

ideator = Agent(
    role="Low-Correlation BRAIN Innovator",
    goal="Create truly innovative, submittable alphas that have low correlation to existing forum templates and common WorldQuant factors.",
    backstory="You are a contrarian quant who recombines master ideas with fresh edges while keeping everything simple enough for a clean BRAIN expression.",
    llm=llm,
    verbose=True
)

coder = Agent(
    role="WorldQuant BRAIN Expression Expert",
    goal="Convert the best idea into a clean, valid, and efficient WorldQuant BRAIN expression.",
    backstory="You are an ex-WorldQuant Brain coder who only writes correct, high-quality BRAIN expressions. You always include proper neutralization, delay, decay, and truncation settings.",
    llm=llm,
    verbose=True
)

validator = Agent(
    role="WorldQuant Submission Validator",
    goal="Ensure the alpha is innovative, low-correlation, and ready to submit. Output ONLY in the exact user-specified format.",
    backstory="You are the final gatekeeper. You check for simulator compatibility, low correlation, and economic soundness. You never add extra text outside the required format.",
    llm=llm,
    verbose=True
)

In [10]:
# ==================================== DEBUG: Test if your forum PDFs are loaded ====================================
print("Testing forum_search tool...")

test_result = forum_search.run("group_mean")

print("\n" + "="*80)
print("TOOL TEST RESULT:")
print(test_result)  # first 1500 chars
print("\n" + "="*80)
print(f"Length of returned text: {len(test_result)} characters")

Testing forum_search tool...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


TOOL TEST RESULT:
# Group 类别函数
    'group_min': {'min_args': 2, 'max_args': 2, 'arg_types': ['expression', 'category']},
    # group_mean(x, w, group)
    'group_mean': {'min_args': 3, 'max_args': 3, 'arg_types': ['expression', 'expression', 'category']},
    'group_median': {'min_args': 2, 'max_args': 2, 'arg_types': ['expression', 'category']},
    'group_max': {'min_args': 2, 'max_args': 2, 'arg_types': ['expression', 'category']},
    'group_rank': {'min_args': 2, 'max_args': 2, 'arg_types': ['expression', 'category']},
    'group_vector_proj': {'min_args': 3, 'max_args': 3, 'arg_types': ['expression', 'expression',
'category']},
    'group_normalize': {'min_args': 2, 'max_args': 5, 'arg_types': ['expression', 'category', 'expression',
'expression', 'expression']},
    'group_extra': {'min_args': 3, 'max_args': 3, 'arg_types': ['expression', 'expression', 'category']},
    'group_backfill': {'min_args': 3, 'max_args': 4, 'arg_types': ['expression', 'expression', 'expression',
'exp

In [11]:
# ====================== TASKS & CREW ======================
task1 = Task(
    description="""
    You MUST use the forum_search tool at least once with at least two keywords (ideally multiple times) to find relevant threads.
    Search for keywords from the user request.
    Extract real master opinions, alpha templates, crowded signals, and specific discussions from the actual forum PDFs.
    Do NOT use general knowledge. Cite the content you retrieve from the tool.
    """,
    expected_output="Structured summary of REAL content from the forum PDFs with direct quotes and PDF/thread references.",
    agent=researcher
)

task2 = Task(
    description="""
    Generate 3–5 genuinely innovative alpha ideas that are different from existing forum templates and your current portfolio.
    Focus on low correlation and economic rationale that can be expressed cleanly in BRAIN.
    """,
    expected_output="Numbered list of 3–5 alpha ideas with short name, hypothesis, and low-correlation justification.",
    agent=ideator
)

task3 = Task(
    description="""
    Take the BEST idea from Task 2 and write a clean, valid WorldQuant BRAIN expression.
    Choose realistic Target Settings (Universe, Neutralization, Delay, Decay, Truncation).
    """,
    expected_output="One complete alpha in the exact user format (Alpha Name + Economic Hypothesis + Target Settings + Full BRAIN Expression).",
    agent=coder
)

task4 = Task(
    description="""
    Act as strict WorldQuant reviewer.
    Critique the alpha from Task 3 for innovation, low correlation, simulator compatibility, and economic sense.
    If needed, improve it slightly.
    THEN output ONLY the final alpha in the EXACT format the user wants:
    
    **Alpha Name:** ...
    **Economic Hypothesis:** ...
    **Target Settings:** Region: ___ | Universe: ___ | Neutralization: ___ | Delay: ___ | Decay: ___ | Truncation: ___
    **Full BRAIN Expression:** ...
    
    Do not add any extra explanation or text outside this format.
    """,
    expected_output="Final alpha in the exact markdown format requested by the user.",
    agent=validator
)

crew = Crew(
    agents=[researcher, ideator, coder, validator],
    tasks=[task1, task2, task3, task4],
    process=Process.sequential,
    verbose=True,
    # tracing=True
)

In [12]:
# ====================== RUN ======================
if __name__ == "__main__":
    result = crew.kickoff(inputs={
    "user_request": "Create innovative low-correlation alphas using ideas from the WorldQuant Brain Consultant Forum"
    })
    print(result)

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.13.0                                                                                        │
│  Latest version:  1.14.2                                                                                        │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: f19449a0-2dcd-44e9-bb7e-b5df141c4dac                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│      You MUST use the forum_search tool at least once with at least two keywords (ideally multiple times) to    │
│  find relevant threads.                                                                                         │
│      Search for keywords from the user request.                                                                 │
│      Extract real master opinions, alpha templates, crowded signals, and specific discussions from the actual   │
│  forum PDFs.                                                                                                    │
│      Do NOT use general knowledge. Cite the content you retrieve from the tool.                                 │
│                                                                                                                 │
│  ID: 9713000f-fa9e-4f1a-a07f-56c2cb64de3e                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: WorldQuant Forum Researcher & Master Analyst                                                            │
│                                                                                                                 │
│  Task:                                                                                                          │
│      You MUST use the forum_search tool at least once with at least two keywords (ideally multiple times) to    │
│  find relevant threads.                                                                                         │
│      Search for keywords from the user request.                                                                 │
│      Extract real master opinions, alpha templates, crowded signals, and specific discussions from the actual   │
│  forum PDFs.                                                                                                    │
│      Do NOT use general knowledge. Cite the content you retrieve from the tool.                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: forum_search                                                                                             │
│  Args: {'query': 'alpha template crowded signal'}                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: WorldQuant Submission Validator                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **Alpha Name:** Volatility-of-Volatility Divergence (VoVD)                                                     │
│  **Economic Hypothesis:** The market heavily trades standard volatility as a risk factor, frequently            │
│  penalizing it. However, the relative change in a stock's volatility regime compared to its peers is widely     │
│  ignored. When a stock's short-term volatility collapses sharply below its recent historical trend—especially   │
│  when ranked against its subindustry peers—it signals "stealth accumulation." The forced sellers have dried     │
│  up, the spread has tightened, and institutional buyers are quietly building positions without aggressively     │
│  moving the price. By fading the acceleration of volatility, we isolate a purely idiosyncratic second-moment    │
│  dynamic that avoids crowded standard reversal trades.                                                          │
│  **Target Settings:** Region: USA | Universe: TOP3000 | Neutralization: Subindustry | Delay: 1 | Decay: 5 |     │
│  Truncation: 0.08                                                                                               │
│  **Full BRAIN Expression:** rank(ts_decay_linear(group_rank(-ts_delta(ts_std_dev(returns, 10), 20),             │
│  subindustry), 5))                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│      Act as strict WorldQuant reviewer.                                                                         │
│      Critique the alpha from Task 3 for innovation, low correlation, simulator compatibility, and economic      │
│  sense.                                                                                                         │
│      If needed, improve it slightly.                                                                            │
│      THEN output ONLY the final alpha in the EXACT format the user wants:                                       │
│                                                                                                                 │
│      **Alpha Name:** ...                                                                                        │
│      **Economic Hypothesis:** ...                                                                               │
│      **Target Settings:** Region: ___ | Universe: ___ | Neutralization: ___ | Delay: ___ | Decay: ___ |         │
│  Truncation: ___                                                                                                │
│      **Full BRAIN Expression:** ...                                                                             │
│                                                                                                                 │
│      Do not add any extra explanation or text outside this format.                                              │
│                                                                                                                 │
│  Agent: WorldQuant Submission Validator                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: f19449a0-2dcd-44e9-bb7e-b5df141c4dac                                                                       │
│  Final Output: **Alpha Name:** Volatility-of-Volatility Divergence (VoVD)                                       │
│  **Economic Hypothesis:** The market heavily trades standard volatility as a risk factor, frequently            │
│  penalizing it. However, the relative change in a stock's volatility regime compared to its peers is widely     │
│  ignored. When a stock's short-term volatility collapses sharply below its recent historical trend—especially   │
│  when ranked against its subindustry peers—it signals "stealth accumulation." The forced sellers have dried     │
│  up, the spread has tightened, and institutional buyers are quietly building positions without aggressively     │
│  moving the price. By fading the acceleration of volatility, we isolate a purely idiosyncratic second-moment    │
│  dynamic that avoids crowded standard reversal trades.                                                          │
│  **Target Settings:** Region: USA | Universe: TOP3000 | Neutralization: Subindustry | Delay: 1 | Decay: 5 |     │
│  Truncation: 0.08                                                                                               │
│  **Full BRAIN Expression:** rank(ts_decay_linear(group_rank(-ts_delta(ts_std_dev(returns, 10), 20),             │
│  subindustry), 5))                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**Alpha Name:** Volatility-of-Volatility Divergence (VoVD)
**Economic Hypothesis:** The market heavily trades standard volatility as a risk factor, frequently penalizing it. However, the relative change in a stock's volatility regime compared to its peers is widely ignored. When a stock's short-term volatility collapses sharply below its recent historical trend—especially when ranked against its subindustry peers—it signals "stealth accumulation." The forced sellers have dried up, the spread has tightened, and institutional buyers are quietly building positions without aggressively moving the price. By fading the acceleration of volatility, we isolate a purely idiosyncratic second-moment dynamic that avoids crowded standard reversal trades.
**Target Settings:** Region: USA | Universe: TOP3000 | Neutralization: Subindustry | Delay: 1 | Decay: 5 | Truncation: 0.08
**Full BRAIN Expression:** rank(ts_decay_linear(group_rank(-ts_delta(ts_std_dev(returns, 10), 20), subindustry), 5))
